# ClipForge · Motor editorial gratuito en Google Colab
Este cuaderno inicia Qwen3 4B en la sesión gratuita de Colab y publica una URL HTTPS temporal protegida por un token. No requiere tarjeta ni clave de API de pago. **No es ilimitado ni permanente**: Colab puede limitar GPU, duración y disponibilidad; la URL y el token cambian al reiniciar.

**Inicio:** abre Runtime → Change runtime type → T4 GPU si está disponible. Ejecuta las celdas en orden. No compartas el token ni la URL.

**Privacidad:** el vídeo y la transcripción permanecen en el navegador; ClipForge envía al motor solo el texto de los candidatos y el contexto necesario para juzgarlos.

In [ ]:
# 1) Comprobar acelerador y herramientas
import subprocess, os, shutil, time, secrets, textwrap, pathlib, json, threading
gpu = subprocess.run(['bash','-lc','nvidia-smi'],capture_output=True,text=True)
print(gpu.stdout[:2500] if gpu.returncode == 0 else 'No hay GPU asignada. Se intentará CPU; será mucho más lento.')
print('CPU:', os.cpu_count(), '· RAM libre aproximada:')
subprocess.run(['bash','-lc','free -h'])


In [ ]:
# 2) Instalar Ollama y Cloudflare Tunnel
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Instalación terminada')


In [ ]:
# 3) Descargar Qwen3 4B y levantar una API compatible con ClipForge
import subprocess, os, time, secrets, pathlib, json, threading, urllib.request, urllib.error
os.environ['OLLAMA_HOST']='127.0.0.1:11434'
ollama = subprocess.Popen(['ollama','serve'],stdout=open('/tmp/ollama.log','w'),stderr=subprocess.STDOUT,env=os.environ.copy())
for _ in range(90):
    if subprocess.run(['bash','-lc','curl -fsS http://127.0.0.1:11434/api/tags >/dev/null'],capture_output=True).returncode == 0: break
    time.sleep(2)
else: raise RuntimeError('Ollama no inició. Revisa /tmp/ollama.log')
MODEL='qwen3:4b'
subprocess.run(['ollama','pull',MODEL],check=True)
TOKEN=secrets.token_urlsafe(32)
proxy = r'''from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import json, urllib.request, os, threading, time
TOKEN=os.environ['CLIPFORGE_TOKEN']; MODEL='qwen3:4b'; LOCK=threading.Lock(); LAST=[0.0]
def ollama_ready():
 try:
  with urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=5) as r: data=json.loads(r.read())
  return any(x.get('name')==MODEL for x in data.get('models',[]))
 except Exception: return False
class H(BaseHTTPRequestHandler):
 def cors(self):
  self.send_header('Access-Control-Allow-Origin','*'); self.send_header('Access-Control-Allow-Headers','Authorization, Content-Type'); self.send_header('Access-Control-Allow-Methods','GET, POST, OPTIONS'); self.send_header('Cache-Control','no-store')
 def do_OPTIONS(self):
  self.send_response(204); self.cors(); self.end_headers()
 def do_GET(self):
  if self.path!='/health' or self.headers.get('Authorization')!='Bearer '+TOKEN:
   self.send_response(401); self.cors(); self.end_headers(); return
  ready=ollama_ready(); body=json.dumps({'status':'ready' if ready else 'loading','model':MODEL if ready else None}).encode()
  self.send_response(200 if ready else 503); self.send_header('Content-Type','application/json'); self.cors(); self.end_headers(); self.wfile.write(body)
 def do_POST(self):
  if self.path!='/v1/chat/completions' or self.headers.get('Authorization')!='Bearer '+TOKEN:
   self.send_response(401); self.cors(); self.end_headers(); return
  try:
   n=int(self.headers.get('Content-Length','0'))
   if n<2 or n>1000000: raise ValueError('Payload inválido o demasiado grande')
   data=json.loads(self.rfile.read(n)); messages=data.get('messages')
   if not isinstance(messages,list) or not messages or sum(len(str(x.get('content',''))) for x in messages)>80000: raise ValueError('Mensajes inválidos o demasiado largos')
   payload={'model':MODEL,'messages':messages,'stream':False,'think':False,'format':'json','options':{'temperature':max(0,min(.3,float(data.get('temperature',.1)) )),'num_ctx':8192,'num_predict':max(128,min(1200,int(data.get('max_tokens',900))))}}
   with LOCK:
    req=urllib.request.Request('http://127.0.0.1:11434/api/chat',data=json.dumps(payload).encode(),headers={'Content-Type':'application/json'},method='POST')
    with urllib.request.urlopen(req,timeout=300) as r: result=json.loads(r.read())
   content=result.get('message',{}).get('content','')
   if not content: raise RuntimeError('Qwen no devolvió contenido')
   out=json.dumps({'choices':[{'index':0,'message':{'role':'assistant','content':content},'finish_reason':'stop'}],'model':MODEL}).encode()
   self.send_response(200); self.send_header('Content-Type','application/json'); self.cors(); self.end_headers(); self.wfile.write(out)
  except Exception as e:
   out=json.dumps({'error':str(e)[:500]}).encode(); self.send_response(502); self.send_header('Content-Type','application/json'); self.cors(); self.end_headers(); self.wfile.write(out)
 def log_message(self,*args): pass
ThreadingHTTPServer(('127.0.0.1',8000),H).serve_forever()
'''
pathlib.Path('/tmp/clipforge_proxy.py').write_text(proxy)
env=os.environ.copy(); env['CLIPFORGE_TOKEN']=TOKEN
server=subprocess.Popen(['python','/tmp/clipforge_proxy.py'],env=env,stdout=open('/tmp/proxy.log','w'),stderr=subprocess.STDOUT)
for _ in range(30):
 try:
  req=urllib.request.Request('http://127.0.0.1:8000/health',headers={'Authorization':'Bearer '+TOKEN})
  with urllib.request.urlopen(req,timeout=5) as r: print('Motor:',r.status,r.read().decode()); break
 except Exception: time.sleep(1)
print('Proxy listo. Token de esta sesión:',TOKEN)


In [ ]:
# 4) Abrir túnel HTTPS temporal
tunnel = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8000'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
import re, threading
lines=[]
def read_tunnel():
    for line in tunnel.stdout: lines.append(line)
threading.Thread(target=read_tunnel,daemon=True).start()
url=None
for _ in range(120):
    m=re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com',''.join(lines))
    if m: url=m.group(0); break
    time.sleep(1)
if not url: raise RuntimeError('No se pudo crear el túnel: '+''.join(lines[-20:]))
print('URL DEL MOTOR:',url)
print('TOKEN:',TOKEN)
print('Copia ambos datos en ClipForge → Motor 02 → Motor editorial IA gratuito en la nube.')
print('Mantén esta sesión de Colab abierta mientras ClipForge analiza. No compartas la URL ni el token.')


In [ ]:
# 5) Prueba real de inferencia antes de conectar ClipForge
payload={'model':MODEL,'messages':[{'role':'system','content':'Responde solo JSON válido.'},{'role':'user','content':'Devuelve exactamente {"ok":true,"model":"qwen3:4b"}'}],'stream':False,'think':False,'format':'json','options':{'num_ctx':2048,'num_predict':100}}
req=urllib.request.Request('http://127.0.0.1:11434/api/chat',data=json.dumps(payload).encode(),headers={'Content-Type':'application/json'},method='POST')
with urllib.request.urlopen(req,timeout=180) as response: test=json.loads(response.read())
print('Respuesta Qwen:',test.get('message',{}).get('content'))
assert 'ok' in test.get('message',{}).get('content','').lower(), 'Qwen no superó la prueba de inferencia JSON'
print('PASS: el modelo respondió. Ahora conecta URL y token en ClipForge.')
